In [3]:
# Colab-ready end-to-end: IMDB -> preprocess -> gensim GloVe 100d -> Dataset -> Collate (pad+pack) -> LSTM train
# Run in one cell. Installs only if missing.

# Install requirements (uncomment in Colab if needed)
# !pip install -q gensim

import re
import time
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
import gensim.downloader as api
from datasets import load_dataset
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# ---------------------------
# 1) Setup / downloads
# ---------------------------
nltk.download("punkt")
nltk.download("wordnet")
nltk.download("stopwords")
nltk.download("punkt_tab")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ---------------------------
# 2) Load small pretrained embeddings (GloVe 100d) via gensim
# ---------------------------
print("Loading embeddings (glove-wiki-gigaword-100) — small & Colab friendly...")
emb_model = api.load("glove-wiki-gigaword-100")   # ~128MB download first time
EMB_DIM = emb_model.vector_size
print("Embedding dim:", EMB_DIM)

# ---------------------------
# 3) Load IMDB dataset (HuggingFace datasets)
# ---------------------------
print("Loading IMDB dataset (train subset for speed)...")
ds = load_dataset("imdb")
# Use smaller subset for quick demo (remove slicing for full dataset)
train_raw = ds["train"]  # 25k samples
test_raw  = ds["test"]

# optional: quick debug subset
# train_raw = train_raw.select(range(2000))
# test_raw = test_raw.select(range(1000))

# ---------------------------
# 4) Preprocessing utilities
# ---------------------------
clean_re = re.compile(r"[^\w\s]")  # remove punctuation but keep unicode word chars
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    """
    Input: raw string
    Output: token list (cleaned + lowercased + lemmatized, stopwords removed)
    Note: keep tokens as words (not subwords) because we use GloVe keyed vectors.
    """
    text = text.lower()
    text = clean_re.sub(" ", text)          # remove punctuation -> spaces (avoid concatenation)
    toks = word_tokenize(text)
    out = []
    for t in toks:
        if t.isnumeric():                   # optional: keep numbers, or skip
            out.append(t)
            continue
        if t in stop_words:
            continue
        lemma = lemmatizer.lemmatize(t)
        out.append(lemma)
    return out

# Pre-tokenize (keep tokens in memory; small dataset still OK)
print("Preprocessing texts (this may take a minute)...")
t0 = time.time()
train_tokens = [preprocess_text(x["text"]) for x in train_raw]
train_labels = [int(x["label"]) for x in train_raw]
test_tokens  = [preprocess_text(x["text"]) for x in test_raw]
test_labels  = [int(x["label"]) for x in test_raw]
print("Preprocessing done in {:.1f}s".format(time.time()-t0))

# ---------------------------
# 5) Dataset wrapper (on-the-fly embedding lookup)
# ---------------------------
class EmbeddingDataset(Dataset):
    def __init__(self, tokenized_texts, labels, embedding_model, emb_dim=100):
        """
        tokenized_texts: list[list[str]]
        labels: list[int]
        embedding_model: gensim KeyedVectors (supports 'word in model' checks and model[word])
        """
        assert len(tokenized_texts) == len(labels)
        self.texts = tokenized_texts
        self.labels = labels
        self.emb = embedding_model
        self.emb_dim = emb_dim
        # precompute zero vector for OOV
        self.zero = np.zeros(self.emb_dim, dtype=np.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = self.texts[idx]
        # Convert tokens -> embeddings (on-the-fly)
        # produce a torch.FloatTensor of shape (seq_len, emb_dim)
        vecs = []
        for w in tokens:
            if w in self.emb:
                v = self.emb[w]           # numpy array
            else:
                # OOV fallback: zero vector (fast). Could use random small vector instead.
                v = self.zero
            vecs.append(torch.from_numpy(np.asarray(v, dtype=np.float32)))
        if len(vecs) == 0:
            # guard: empty sentence -> single zero token (seq_len=1)
            vecs = [torch.from_numpy(self.zero.copy())]
        seq_tensor = torch.stack(vecs)   # (seq_len, emb_dim)
        label_tensor = torch.tensor(self.labels[idx], dtype=torch.long)
        return seq_tensor, label_tensor

# make dataset
train_dataset = EmbeddingDataset(train_tokens, train_labels, emb_model, EMB_DIM)
test_dataset  = EmbeddingDataset(test_tokens, test_labels, emb_model, EMB_DIM)

# ---------------------------
# 6) collate_fn: pad + sort + pack
# ---------------------------
def collate_pad_pack(batch):
    """
    batch: list of (seq_tensor (seq_len, emb_dim), label)
    returns: packed_sequence, labels_sorted (tensor)
    """
    sequences, labels = zip(*batch)
    lengths = torch.tensor([seq.size(0) for seq in sequences], dtype=torch.long)
    # sort descending
    lengths_sorted, indices = torch.sort(lengths, descending=True)
    sequences_sorted = [sequences[i] for i in indices]
    labels_sorted = torch.tensor([labels[i] for i in indices], dtype=torch.long)
    # pad
    padded = pad_sequence(sequences_sorted, batch_first=True)  # (B, L, emb_dim)
    # pack
    packed = pack_padded_sequence(padded, lengths_sorted.cpu(), batch_first=True, enforce_sorted=True)
    return packed, labels_sorted

# DataLoaders
BATCH = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH, shuffle=True, collate_fn=collate_pad_pack)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH, shuffle=False, collate_fn=collate_pad_pack)

# ---------------------------
# 7) LSTM model that accepts packed sequence
# ---------------------------
class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=1, bidirectional=False, dropout=0.2):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1

        # Note: no embedding layer here because we pass embeddings directly
        self.lstm = nn.LSTM(input_size=input_dim,
                            hidden_size=hidden_dim,
                            num_layers=num_layers,
                            batch_first=True,
                            bidirectional=bidirectional)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * self.num_directions, output_dim)

    def forward(self, packed_input):
        # Accepts a PackedSequence
        packed_out, (h_n, c_n) = self.lstm(packed_input)
        # h_n shape: (num_layers * num_directions, batch, hidden_dim)
        # take last layer's hidden states
        last = h_n.view(self.num_layers, self.num_directions, h_n.size(1), self.hidden_dim)[-1]
        # last shape = (num_directions, batch, hidden_dim)
        if self.num_directions == 2:
            last = torch.cat([last[0], last[1]], dim=1)  # (batch, hidden_dim*2)
        else:
            last = last.squeeze(0)  # (batch, hidden_dim)
        out = self.dropout(last)
        logits = self.classifier(out)
        return logits

# ---------------------------
# 8) Training boilerplate
# ---------------------------
model = LSTMClassifier(input_dim=EMB_DIM, hidden_dim=128, output_dim=2, num_layers=1, bidirectional=True).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    for packed, labels in loader:
        # packed is a PackedSequence with data on CPU; move its data to device
        # trick: packed.data is (sum_len, emb_dim) -> move then rebuild PackedSequence
        data_on_device = packed.data.to(device)
        packed = torch.nn.utils.rnn.PackedSequence(data_on_device, packed.batch_sizes, packed.sorted_indices, packed.unsorted_indices)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(packed)   # shape: (batch, num_classes)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)
    avg_loss = total_loss / total_samples
    acc = total_correct / total_samples
    return avg_loss, acc

def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    with torch.no_grad():
        for packed, labels in loader:
            data_on_device = packed.data.to(device)
            packed = torch.nn.utils.rnn.PackedSequence(data_on_device, packed.batch_sizes, packed.sorted_indices, packed.unsorted_indices)
            labels = labels.to(device)
            logits = model(packed)
            loss = criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)
            preds = logits.argmax(dim=1)
            total_correct += (preds == labels).sum().item()
            total_samples += labels.size(0)
    avg_loss = total_loss / total_samples
    acc = total_correct / total_samples
    return avg_loss, acc

# ---------------------------
# 9) Quick train-run (1 epoch for demo)
# ---------------------------
EPOCHS = 10  # bump up for real training
print("Starting training...")
t0 = time.time()
for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = eval_epoch(model, test_loader, criterion, device)
    print(f"Epoch {epoch+1}/{EPOCHS} — train loss {train_loss:.4f} acc {train_acc:.4f}; val loss {val_loss:.4f} acc {val_acc:.4f}")
print("Done in {:.1f}s".format(time.time()-t0))


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Device: cuda
Loading embeddings (glove-wiki-gigaword-100) — small & Colab friendly...
Embedding dim: 100
Loading IMDB dataset (train subset for speed)...
Preprocessing texts (this may take a minute)...
Preprocessing done in 65.7s
Starting training...
Epoch 1/10 — train loss 0.4916 acc 0.7586; val loss 0.4182 acc 0.8118
Epoch 2/10 — train loss 0.3875 acc 0.8299; val loss 0.3528 acc 0.8470
Epoch 3/10 — train loss 0.3278 acc 0.8619; val loss 0.3094 acc 0.8677
Epoch 4/10 — train loss 0.2965 acc 0.8769; val loss 0.2942 acc 0.8736
Epoch 5/10 — train loss 0.2619 acc 0.8926; val loss 0.2936 acc 0.8758
Epoch 6/10 — train loss 0.2291 acc 0.9076; val loss 0.3138 acc 0.8784
Epoch 7/10 — train loss 0.1977 acc 0.9214; val loss 0.3243 acc 0.8679
Epoch 8/10 — train loss 0.1529 acc 0.9402; val loss 0.3386 acc 0.8769
Epoch 9/10 — train loss 0.1139 acc 0.9579; val loss 0.3701 acc 0.8728
Epoch 10/10 — train loss 0.0787 acc 0.9712; val loss 0.4348 acc 0.8655
Done in 532.6s
